## 1. Importar librerías

In [22]:
import os
import glob
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

## 2. Definir rutas

In [23]:
RAW_DIR = "../data/raw"
PROCESSED_DIR = "../data/processed"

os.makedirs(PROCESSED_DIR, exist_ok=True)

files = glob.glob(os.path.join(RAW_DIR, "*.csv"))

print("Archivos encontrados:")
for file in files:
    print("-", os.path.basename(file))

Archivos encontrados:
- Data_hasta_13-07.csv
- Data_radiacion_-2025_08_29.csv
- Data_rad_y_temp_20-08_26-08-2025.csv
- Data_temperatura.csv
- Hasta_el_25_07_25.csv


## 3. Función para cargar archivos

In [24]:
def load_file(path):
    # Carga archivos CSV meteorológicos
    df = pd.read_csv(path)

    df["archivo_origen"] = os.path.basename(path)
    return df

## 4. Función para estandarizar columnas

In [25]:
def standardize_columns(df):
    # Estandariza nombres de columnas meteorológicas.
    rename_dict = {}

    for col in df.columns:
        col_lower = col.lower()

        if col_lower == "date":
            rename_dict[col] = "fecha_hora"

        elif "temperature" in col_lower:
            rename_dict[col] = "temperatura"

        elif "solar radiation" in col_lower:
            rename_dict[col] = "radiacion"

        elif col_lower.startswith("rh") or "relative humidity" in col_lower:
            rename_dict[col] = "humedad"

        elif "wind speed" in col_lower:
            rename_dict[col] = "viento"

        elif "gust speed" in col_lower:
            rename_dict[col] = "rafaga"

        elif col_lower == "line#":
            rename_dict[col] = "linea"

    df = df.rename(columns=rename_dict)

    useful_columns = [
        "fecha_hora",
        "temperatura",
        "radiacion",
        "humedad",
        "viento",
        "rafaga",
        "archivo_origen"
    ]

    existing_columns = [col for col in useful_columns if col in df.columns]
    df = df[existing_columns]

    return df

## 5. Cargar, limpiar y unir todos los archivos

In [26]:
dataframes = []

for file in files:
    df_temp = load_file(file)
    df_temp = standardize_columns(df_temp)
    dataframes.append(df_temp)

df_raw = pd.concat(dataframes, ignore_index=True)

print("Dimensiones iniciales:", df_raw.shape)
df_raw.head()

Dimensiones iniciales: (60124, 7)


,fecha_hora,temperatura,radiacion,humedad,archivo_origen,viento,rafaga
0,25-06-18 15:40:00 -0500,12.196,124,53.196,Data_hasta_13-07.csv,NaN,NaN
1,25-06-18 15:45:00 -0500,12.260,123,55.476,Data_hasta_13-07.csv,NaN,NaN
2,25-06-18 15:50:00 -0500,11.939,117,55.155,Data_hasta_13-07.csv,NaN,NaN
3,25-06-18 15:55:00 -0500,11.746,109,55.888,Data_hasta_13-07.csv,NaN,NaN
4,25-06-18 16:00:00 -0500,11.617,103,56.510,Data_hasta_13-07.csv,NaN,NaN


## 6. Convertir fechas y limpiar errores

In [27]:
df = df_raw.copy()

# Reemplazar textos de error por NaN
df = df.replace(["ERROR", "Error", "error", "", " ", "--", "NaN"], np.nan)

# Convertir fecha
df["fecha_hora"] = pd.to_datetime(
    df["fecha_hora"],
    format="%y-%m-%d %H:%M:%S %z",
    errors="coerce"
)

# Quitar zona horaria para trabajar más fácil
df["fecha_hora"] = df["fecha_hora"].dt.tz_convert(None)

# Convertir variables numéricas
numeric_columns = ["temperatura", "radiacion", "humedad", "viento", "rafaga"]

for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Eliminar filas sin fecha
df = df.dropna(subset=["fecha_hora"])

df = df.sort_values("fecha_hora").reset_index(drop=True)

print("Dimensiones después de limpiar fechas:", df.shape)
df.head()

Dimensiones después de limpiar fechas: (60124, 7)


,fecha_hora,temperatura,radiacion,humedad,archivo_origen,viento,rafaga
0,2025-06-18 20:40:00,12.196,124.0,53.196,Data_hasta_13-07.csv,NaN,NaN
1,2025-06-18 20:45:00,12.260,123.0,55.476,Data_hasta_13-07.csv,NaN,NaN
2,2025-06-18 20:50:00,11.939,117.0,55.155,Data_hasta_13-07.csv,NaN,NaN
3,2025-06-18 20:55:00,11.746,109.0,55.888,Data_hasta_13-07.csv,NaN,NaN
4,2025-06-18 21:00:00,11.617,103.0,56.510,Data_hasta_13-07.csv,NaN,NaN


## 7. Unir registros repetidos por fecha

In [28]:
df_master = (
    df
    .groupby("fecha_hora", as_index=False)
    .agg({
        "temperatura": "mean",
        "radiacion": "mean",
        "humedad": "mean",
        "viento": "mean",
        "rafaga": "mean"
    })
)

df_master = df_master.sort_values("fecha_hora").reset_index(drop=True)

print("Dataset maestro inicial:", df_master.shape)
df_master.head()

Dataset maestro inicial: (20237, 6)


,fecha_hora,temperatura,radiacion,humedad,viento,rafaga
0,2025-06-18 20:40:00,12.196,124.0,53.196,NaN,NaN
1,2025-06-18 20:45:00,12.260,123.0,55.476,NaN,NaN
2,2025-06-18 20:50:00,11.939,117.0,55.155,NaN,NaN
3,2025-06-18 20:55:00,11.746,109.0,55.888,NaN,NaN
4,2025-06-18 21:00:00,11.617,103.0,56.510,NaN,NaN


## 8. Revisar valores faltantes

In [29]:
missing_report = df_master.isna().mean().sort_values(ascending=False) * 100

print("Porcentaje de valores faltantes:")
print(missing_report)

Porcentaje de valores faltantes:
humedad        66.842911
viento         45.283392
rafaga         45.283392
radiacion      29.470771
temperatura     0.000000
fecha_hora      0.000000
dtype: float64


## 9. Imputar valores faltantes

In [30]:
df_master = df_master.set_index("fecha_hora")

# Interpolación temporal
for col in ["temperatura", "radiacion", "humedad", "viento", "rafaga"]:
    if col in df_master.columns:
        df_master[col] = df_master[col].interpolate(method="time", limit_direction="both")

# Si todavía quedan NaN, completar con mediana
for col in ["temperatura", "radiacion", "humedad", "viento", "rafaga"]:
    if col in df_master.columns:
        df_master[col] = df_master[col].fillna(df_master[col].median())

df_master = df_master.reset_index()

print("Valores faltantes después de imputar:")
print(df_master.isna().sum())

Valores faltantes después de imputar:
fecha_hora     0
temperatura    0
radiacion      0
humedad        0
viento         0
rafaga         0
dtype: int64


## 10. Crear variables temporales

In [31]:
df_master["hora"] = df_master["fecha_hora"].dt.hour
df_master["dia"] = df_master["fecha_hora"].dt.day
df_master["mes"] = df_master["fecha_hora"].dt.month
df_master["dia_semana"] = df_master["fecha_hora"].dt.dayofweek
df_master["dia_anio"] = df_master["fecha_hora"].dt.dayofyear

# Variables cíclicas para hora y mes
df_master["hora_sin"] = np.sin(2 * np.pi * df_master["hora"] / 24)
df_master["hora_cos"] = np.cos(2 * np.pi * df_master["hora"] / 24)

df_master["mes_sin"] = np.sin(2 * np.pi * df_master["mes"] / 12)
df_master["mes_cos"] = np.cos(2 * np.pi * df_master["mes"] / 12)

df_master.head()

,fecha_hora,temperatura,radiacion,humedad,viento,rafaga,hora,dia,mes,dia_semana,dia_anio,hora_sin,hora_cos,mes_sin,mes_cos
0,2025-06-18 20:40:00,12.196,124.0,53.196,1.0,1.3,20,18,6,2,169,-0.866025,0.500000,1.224647e-16,-1.0
1,2025-06-18 20:45:00,12.260,123.0,55.476,1.0,1.3,20,18,6,2,169,-0.866025,0.500000,1.224647e-16,-1.0
2,2025-06-18 20:50:00,11.939,117.0,55.155,1.0,1.3,20,18,6,2,169,-0.866025,0.500000,1.224647e-16,-1.0
3,2025-06-18 20:55:00,11.746,109.0,55.888,1.0,1.3,20,18,6,2,169,-0.866025,0.500000,1.224647e-16,-1.0
4,2025-06-18 21:00:00,11.617,103.0,56.510,1.0,1.3,21,18,6,2,169,-0.707107,0.707107,1.224647e-16,-1.0


## 11. Detectar frecuencia temporal

In [32]:
df_master = df_master.sort_values("fecha_hora").reset_index(drop=True)

median_delta = df_master["fecha_hora"].diff().median()
minutes_per_step = median_delta.total_seconds() / 60

print("Frecuencia aproximada de los datos:", minutes_per_step, "minutos")

steps_per_hour = int(round(60 / minutes_per_step))
steps_24h = int(round(24 * 60 / minutes_per_step))

print("Registros por hora:", steps_per_hour)
print("Registros en 24 horas:", steps_24h)

Frecuencia aproximada de los datos: 5.0 minutos
Registros por hora: 12
Registros en 24 horas: 288


## 12. Crear rezagos térmicos

In [33]:
lag_hours = [1, 3, 6, 12, 24]

for h in lag_hours:
    lag_steps = h * steps_per_hour
    df_master[f"temp_lag_{h}h"] = df_master["temperatura"].shift(lag_steps)
    df_master[f"radiacion_lag_{h}h"] = df_master["radiacion"].shift(lag_steps)

    if "humedad" in df_master.columns:
        df_master[f"humedad_lag_{h}h"] = df_master["humedad"].shift(lag_steps)

    if "viento" in df_master.columns:
        df_master[f"viento_lag_{h}h"] = df_master["viento"].shift(lag_steps)

df_master.head()

,fecha_hora,temperatura,radiacion,humedad,viento,rafaga,hora,dia,mes,dia_semana,...,humedad_lag_6h,viento_lag_6h,temp_lag_12h,radiacion_lag_12h,humedad_lag_12h,viento_lag_12h,temp_lag_24h,radiacion_lag_24h,humedad_lag_24h,viento_lag_24h
0,2025-06-18 20:40:00,12.196,124.0,53.196,1.0,1.3,20,18,6,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-06-18 20:45:00,12.260,123.0,55.476,1.0,1.3,20,18,6,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-06-18 20:50:00,11.939,117.0,55.155,1.0,1.3,20,18,6,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-06-18 20:55:00,11.746,109.0,55.888,1.0,1.3,20,18,6,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-06-18 21:00:00,11.617,103.0,56.510,1.0,1.3,21,18,6,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 13. Crear gradientes térmicos

In [34]:
for h in [1, 3, 6, 12]:
    df_master[f"gradiente_temp_{h}h"] = df_master["temperatura"] - df_master[f"temp_lag_{h}h"]

df_master.head()

,fecha_hora,temperatura,radiacion,humedad,viento,rafaga,hora,dia,mes,dia_semana,...,humedad_lag_12h,viento_lag_12h,temp_lag_24h,radiacion_lag_24h,humedad_lag_24h,viento_lag_24h,gradiente_temp_1h,gradiente_temp_3h,gradiente_temp_6h,gradiente_temp_12h
0,2025-06-18 20:40:00,12.196,124.0,53.196,1.0,1.3,20,18,6,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-06-18 20:45:00,12.260,123.0,55.476,1.0,1.3,20,18,6,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-06-18 20:50:00,11.939,117.0,55.155,1.0,1.3,20,18,6,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-06-18 20:55:00,11.746,109.0,55.888,1.0,1.3,20,18,6,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-06-18 21:00:00,11.617,103.0,56.510,1.0,1.3,21,18,6,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 14. Crear variables móviles

In [35]:
rolling_hours = [3, 6, 12, 24]

for h in rolling_hours:
    window = h * steps_per_hour

    df_master[f"temp_media_{h}h"] = df_master["temperatura"].rolling(window=window).mean()
    df_master[f"temp_min_{h}h"] = df_master["temperatura"].rolling(window=window).min()
    df_master[f"temp_max_{h}h"] = df_master["temperatura"].rolling(window=window).max()

    df_master[f"radiacion_media_{h}h"] = df_master["radiacion"].rolling(window=window).mean()

    if "humedad" in df_master.columns:
        df_master[f"humedad_media_{h}h"] = df_master["humedad"].rolling(window=window).mean()

    if "viento" in df_master.columns:
        df_master[f"viento_media_{h}h"] = df_master["viento"].rolling(window=window).mean()

df_master.head()

,fecha_hora,temperatura,radiacion,humedad,viento,rafaga,hora,dia,mes,dia_semana,...,temp_max_12h,radiacion_media_12h,humedad_media_12h,viento_media_12h,temp_media_24h,temp_min_24h,temp_max_24h,radiacion_media_24h,humedad_media_24h,viento_media_24h
0,2025-06-18 20:40:00,12.196,124.0,53.196,1.0,1.3,20,18,6,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-06-18 20:45:00,12.260,123.0,55.476,1.0,1.3,20,18,6,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-06-18 20:50:00,11.939,117.0,55.155,1.0,1.3,20,18,6,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-06-18 20:55:00,11.746,109.0,55.888,1.0,1.3,20,18,6,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-06-18 21:00:00,11.617,103.0,56.510,1.0,1.3,21,18,6,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 15. Crear variable objetivo helada_24h

In [36]:
df_master["temperatura_24h_futura"] = df_master["temperatura"].shift(-steps_24h)

df_master["helada_24h"] = (df_master["temperatura_24h_futura"] <= 0).astype(int)

df_master[["fecha_hora", "temperatura", "temperatura_24h_futura", "helada_24h"]].head()

,fecha_hora,temperatura,temperatura_24h_futura,helada_24h
0,2025-06-18 20:40:00,12.196,-4.984,1
1,2025-06-18 20:45:00,12.260,-5.284,1
2,2025-06-18 20:50:00,11.939,-6.163,1
3,2025-06-18 20:55:00,11.746,-5.970,1
4,2025-06-18 21:00:00,11.617,-5.434,1


## 16. Eliminar filas incompletas por rezagos y objetivo futuro

In [37]:
df_model = df_master.copy()

# Eliminar filas donde no se pudo calcular rezagos o temperatura futura
df_model = df_model.dropna().reset_index(drop=True)

print("Dataset final para modelado:", df_model.shape)

print("Distribución de la variable objetivo:")
print(df_model["helada_24h"].value_counts())
print(df_model["helada_24h"].value_counts(normalize=True) * 100)

Dataset final para modelado: (19661, 65)
Distribución de la variable objetivo:
helada_24h
0    13049
1     6612
Name: count, dtype: int64
helada_24h
0    66.369971
1    33.630029
Name: proportion, dtype: float64


## 17. Eliminar columna auxiliar futura

In [38]:
df_model = df_model.drop(columns=["temperatura_24h_futura"])

df_model.head()

,fecha_hora,temperatura,radiacion,humedad,viento,rafaga,hora,dia,mes,dia_semana,...,radiacion_media_12h,humedad_media_12h,viento_media_12h,temp_media_24h,temp_min_24h,temp_max_24h,radiacion_media_24h,humedad_media_24h,viento_media_24h,helada_24h
0,2025-06-21 10:20:00,-4.984,1.0,98.128,1.0,1.3,10,21,6,5,...,1.020833,91.478417,1.0,9.468552,-6.271,30.127,179.850694,63.675063,1.0,1
1,2025-06-21 10:25:00,-5.284,1.0,97.961,1.0,1.3,10,21,6,5,...,1.006944,91.736611,1.0,9.407635,-6.271,30.127,179.427083,63.822580,1.0,1
2,2025-06-21 10:30:00,-6.163,1.0,97.913,1.0,1.3,10,21,6,5,...,1.000000,91.988187,1.0,9.344781,-6.271,30.127,179.024306,63.971045,1.0,1
3,2025-06-21 10:35:00,-5.970,1.0,98.044,1.0,1.3,10,21,6,5,...,1.000000,92.234472,1.0,9.283267,-6.271,30.127,178.649306,64.117420,1.0,1
4,2025-06-21 10:40:00,-5.434,1.0,97.908,1.0,1.3,10,21,6,5,...,1.000000,92.474028,1.0,9.224063,-6.271,30.127,178.295139,64.261163,1.0,1


## 18. Guardar dataset maestro

In [39]:
dataset_maestro_path = os.path.join(PROCESSED_DIR, "dataset_maestro.csv")

df_model.to_csv(dataset_maestro_path, index=False)

print("Dataset maestro guardado en:", dataset_maestro_path)

Dataset maestro guardado en: ../data/processed\dataset_maestro.csv


## 19. División temporal train / validation / test

In [40]:
df_model = df_model.sort_values("fecha_hora").reset_index(drop=True)

n = len(df_model)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df_model.iloc[:train_end].copy()
validation_df = df_model.iloc[train_end:val_end].copy()
test_df = df_model.iloc[val_end:].copy()

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)

print("\nRangos de fechas:")
print("Train:", train_df["fecha_hora"].min(), "→", train_df["fecha_hora"].max())
print("Validation:", validation_df["fecha_hora"].min(), "→", validation_df["fecha_hora"].max())
print("Test:", test_df["fecha_hora"].min(), "→", test_df["fecha_hora"].max())

Train: (13762, 64)
Validation: (2949, 64)
Test: (2950, 64)

Rangos de fechas:
Train: 2025-06-21 10:20:00 → 2025-08-08 05:05:00
Validation: 2025-08-08 05:10:00 → 2025-08-18 10:50:00
Test: 2025-08-18 10:55:00 → 2025-08-28 16:45:00


## 20. Guardar train, validation y test

In [41]:
train_path = os.path.join(PROCESSED_DIR, "train.csv")
validation_path = os.path.join(PROCESSED_DIR, "validation.csv")
test_path = os.path.join(PROCESSED_DIR, "test.csv")

train_df.to_csv(train_path, index=False)
validation_df.to_csv(validation_path, index=False)
test_df.to_csv(test_path, index=False)

print("Archivos guardados:")
print("-", train_path)
print("-", validation_path)
print("-", test_path)

Archivos guardados:
- ../data/processed\train.csv
- ../data/processed\validation.csv
- ../data/processed\test.csv


## 21. Resumen final del preprocesamiento

In [42]:
print("Resumen final")
print("=" * 50)

print("Dataset maestro:", df_model.shape)
print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)

print("\nColumnas finales:")
print(df_model.columns.tolist())

print("\nDistribución de helada_24h en train:")
print(train_df["helada_24h"].value_counts(normalize=True) * 100)

print("\nDistribución de helada_24h en validation:")
print(validation_df["helada_24h"].value_counts(normalize=True) * 100)

print("\nDistribución de helada_24h en test:")
print(test_df["helada_24h"].value_counts(normalize=True) * 100)

Resumen final
Dataset maestro: (19661, 64)
Train: (13762, 64)
Validation: (2949, 64)
Test: (2950, 64)

Columnas finales:
['fecha_hora', 'temperatura', 'radiacion', 'humedad', 'viento', 'rafaga', 'hora', 'dia', 'mes', 'dia_semana', 'dia_anio', 'hora_sin', 'hora_cos', 'mes_sin', 'mes_cos', 'temp_lag_1h', 'radiacion_lag_1h', 'humedad_lag_1h', 'viento_lag_1h', 'temp_lag_3h', 'radiacion_lag_3h', 'humedad_lag_3h', 'viento_lag_3h', 'temp_lag_6h', 'radiacion_lag_6h', 'humedad_lag_6h', 'viento_lag_6h', 'temp_lag_12h', 'radiacion_lag_12h', 'humedad_lag_12h', 'viento_lag_12h', 'temp_lag_24h', 'radiacion_lag_24h', 'humedad_lag_24h', 'viento_lag_24h', 'gradiente_temp_1h', 'gradiente_temp_3h', 'gradiente_temp_6h', 'gradiente_temp_12h', 'temp_media_3h', 'temp_min_3h', 'temp_max_3h', 'radiacion_media_3h', 'humedad_media_3h', 'viento_media_3h', 'temp_media_6h', 'temp_min_6h', 'temp_max_6h', 'radiacion_media_6h', 'humedad_media_6h', 'viento_media_6h', 'temp_media_12h', 'temp_min_12h', 'temp_max_12h', 'r